# 2. Naive Bayes mit TF-IDF (Klassischer Algorithmus)

In diesem Notebook trainieren wir ein Naive Bayes Modell mit TF-IDF Features als Baseline.


In [4]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

# Add src to path
BASE_DIR = Path().resolve()
sys.path.insert(0, str(BASE_DIR / "src"))
from utils_imdb import read_imdb_split, basic_clean

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


## 2.1 Daten laden und vorbereiten


In [5]:
DATA_ROOT = BASE_DIR / "data" / "aclImdb"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data (set to None for full dataset)
MAX_DOCS_PER_CLASS = None

print("Loading data...")
train = read_imdb_split(DATA_ROOT / "train", max_docs_per_class=MAX_DOCS_PER_CLASS)
test = read_imdb_split(DATA_ROOT / "test", max_docs_per_class=MAX_DOCS_PER_CLASS)

# Preprocess
X_train = train["text"].apply(basic_clean)
X_test = test["text"].apply(basic_clean)
y_train, y_test = train["label"], test["label"]

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"Train label distribution:\n{y_train.value_counts()}")


Loading data...


Reading pos: 0it [00:00, ?it/s]
Reading neg: 0it [00:00, ?it/s]
Reading pos: 0it [00:00, ?it/s]
Reading neg: 100%|████████████████████████| 5706/5706 [00:05<00:00, 1064.14it/s]


Train size: 0
Test size: 5706
Train label distribution:
Series([], Name: count, dtype: int64)


## 2.2 Modell definieren und trainieren


In [6]:
# TF-IDF + Naive Bayes Pipeline
clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),  # Unigrams and bigrams
        min_df=2,  # Minimum document frequency
        strip_accents="unicode",
    )),
    ("nb", MultinomialNB(alpha=0.5)),  # Laplace smoothing
])

print("Training model...")
clf.fit(X_train, y_train)
print("Training completed!")


Training model...


ValueError: empty vocabulary; perhaps the documents only contain stop words

## 2.3 Evaluation


In [ ]:
# Predictions
pred = clf.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, pred)
print(cm)


In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
ax.set_title('Naive Bayes + TF-IDF - Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_nb.png", dpi=300, bbox_inches='tight')
plt.show()


## 2.4 Modell und Predictions speichern


In [ ]:
# Save model
models_dir = OUTPUT_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "nb_tfidf_imdb.joblib"
joblib.dump(clf, model_path)
print(f"Model saved: {model_path}")

# Save predictions
preds_path = OUTPUT_DIR / "preds_nb.csv"
pd.DataFrame({
    "text": X_test,
    "y_true": y_test,
    "y_pred": pred
}).to_csv(preds_path, index=False, encoding="utf-8")
print(f"Predictions saved: {preds_path}")
